# Código para calcular los centros de gravedad por comuna en Chile

In [19]:
#!pip install pyarrow

In [80]:
# imports
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from shapely.geometry import LineString
import networkx as nx
from scipy.spatial import cKDTree

In [81]:
## Data 

## data censo 2024 - manzanas (geoparquet) 
gdf = gpd.read_parquet("/Users/Angelo/Dropbox/Tesis 2026 ME/Codes/centros de gravedad poblacional/Cartografia_censo2024_Pais/Cartografia_censo2024_Pais_Manzanas.parquet")
##  entidad (geopa  rquet)
gdf2 = gpd.read_parquet("/Users/Angelo/Dropbox/Tesis 2026 ME/Codes/centros de gravedad poblacional/Cartografia_censo2024_Pais/Cartografia_censo2024_Pais_Entidades.parquet")

## nodes
nodes = gpd.read_file('/Users/Angelo/Dropbox/VS codes/final output/df_nodes_all.shp') ## red actual 
nodes_1970 = gpd.read_file('/Users/Angelo/Dropbox/VS codes/final output/nodes_1970_unificado.shp') ## red 1970 
nodes_1980 = gpd.read_file('/Users/Angelo/Dropbox/VS codes/final output/nodes_1980_1986_unificado.shp') ## red 1980-1986

## edges
edges = gpd.read_file('/Users/Angelo/Dropbox/VS codes/final output/df_edges_all.shp')
edges_1970 = gpd.read_file('/Users/Angelo/Dropbox/VS codes/final output/edges_1970_unificado.shp')
edges_1980 = gpd.read_file('/Users/Angelo/Dropbox/VS codes/final output/edges_1980_1986_unificado.shp')

### Tratamiento previo de redes: verificación y solución de islas de conectividad 1970 y 1980.

In [ ]:
## Corrección de red 1980: caminos que no se clasificaron como existentes en 1980 pero si aparecen en los mapas.



edges_to_inject_ids = [
    '031089', '031116', '0377', '03254', '031184', '031186', '03159', '03291', '039', '031188',
    '13886', '13884', '1310', '13877', '13876', '139', '11450', '11211' , '11439', '063075', '061588', '063078', '13182',
    '131129', '131117', '13877', '13886', '13876', '139', '1310', '13884', '131649', '1345',
    '131684', '13726', '062259', '062256', '061228', '062259', '062256', '1399999', '052565'
]

# 1. Extraer edges de 1970
extracted_edges = edges_1970[edges_1970['edge_ID'].astype(str).isin(edges_to_inject_ids)].copy()

# 2. Pegar en edges_1980 si no existen
existing_edge_ids_80 = set(edges_1980['edge_ID'].astype(str))
new_edges = extracted_edges[~extracted_edges['edge_ID'].astype(str).isin(existing_edge_ids_80)]

if not new_edges.empty:
    edges_1980 = pd.concat([edges_1980, new_edges], ignore_index=True)
    print(f"Injectados {len(new_edges)} edges de 1970 en 1980.")

# 3. Identificar nodos necesarios (usando node_sta_1 y node_end_f que tienen el prefijo)
needed_node_ids = set(extracted_edges['node_sta_1'].astype(str)).union(set(extracted_edges['node_end_f'].astype(str)))
existing_node_ids_80 = set(nodes_1980['node_ID'].astype(str))
missing_node_ids = needed_node_ids - existing_node_ids_80

if missing_node_ids:
    # Buscar en nodes_1970 usando la columna node_ID (que tiene el prefijo)
    missing_nodes = nodes_1970[nodes_1970['node_ID'].astype(str).isin(missing_node_ids)].copy()
    if not missing_nodes.empty:
        nodes_1980 = pd.concat([nodes_1980, missing_nodes], ignore_index=True)
        print(f"Injectados {len(missing_nodes)} nodos de 1970 en 1980 (usando node_ID con prefijo).")
    else:
        print("No se encontraron los nodos en nodes_1970 con los IDs proporcionados.")
else:
    print("Todos los nodos necesarios ya existen en 1980.")


Injectados 33 edges de 1970 en 1980.
Injectados 11 nodos de 1970 en 1980 (usando node_ID con prefijo).


In [83]:
## Inyección de edges y nodes de 1980 hacia 1970 (Versión invertida)
# Aquí debes poner los IDs de los tramos de 1980 que quieres llevarte a 1970
edges_to_inject_ids_80 = [ '14447' ,'141500', '061594', '061593', '061590', '061428', '061431', '061430' 
    #  'ID_1', 'ID_2', ... (añadir aquí los IDs que detectaste en el reporte de outliers)
]

# 1. Extraer edges de 1980
extracted_edges_80 = edges_1980[edges_1980['edge_ID'].astype(str).isin(edges_to_inject_ids_80)].copy()

# 2. Pegar en edges_1970 si no existen
existing_edge_ids_70 = set(edges_1970['edge_ID'].astype(str))
new_edges_for_70 = extracted_edges_80[~extracted_edges_80['edge_ID'].astype(str).isin(existing_edge_ids_70)]

if not new_edges_for_70.empty:
    edges_1970 = pd.concat([edges_1970, new_edges_for_70], ignore_index=True)
    print(f"✅ Inyectados {len(new_edges_for_70)} edges de 1980 en 1970.")
else:
    print("No se encontraron edges nuevos para inyectar o el ID no coincide.")

# 3. Identificar nodos necesarios de la red 1980 que no existen en 1970
# (Usamos node_sta_1 y node_end_f o node_start/node_end según tus columnas)
s_col = 'node_sta_1' if 'node_sta_1' in extracted_edges_80.columns else 'node_start'
e_col = 'node_end_f' if 'node_end_f' in extracted_edges_80.columns else 'node_end'

needed_node_ids = set(extracted_edges_80[s_col].astype(str)).union(set(extracted_edges_80[e_col].astype(str)))
existing_node_ids_70 = set(nodes_1970['node_ID'].astype(str))
missing_node_ids_70 = needed_node_ids - existing_node_ids_70

if missing_node_ids_70:
    # Buscar en nodes_1980 para traerlos a 1970
    missing_nodes_to_add = nodes_1980[nodes_1980['node_ID'].astype(str).isin(missing_node_ids_70)].copy()
    if not missing_nodes_to_add.empty:
        nodes_1970 = pd.concat([nodes_1970, missing_nodes_to_add], ignore_index=True)
        print(f"✅ Inyectados {len(missing_nodes_to_add)} nodos de 1980 en 1970.")
    else:
        print("⚠️ No se encontraron los nodos en nodes_1980 para completar la conexión.")
else:
    print("Todos los nodos necesarios ya existen en 1970.")


✅ Inyectados 8 edges de 1980 en 1970.
Todos los nodos necesarios ya existen en 1970.


### Estimación Cetnros de gravedad poblacional

In [ ]:

gdf["COMUNA"]  = gdf["COMUNA"].astype(str).str.strip().str.upper()
gdf["REGION"]  = gdf["COD_REGION"].astype(str).str.strip().str.upper()

gdf2["COMUNA"] = gdf2["COMUNA"].astype(str).str.strip().str.upper()
gdf2["REGION"] = gdf2["COD_REGION"].astype(str).str.strip().str.upper()

# 1. CENTROIDES POBLACIONALES (MANZANAS)
gdf["centroid"] = gdf.geometry.centroid
gdf["x"] = gdf["centroid"].x
gdf["y"] = gdf["centroid"].y

gdf["x_pop"] = gdf["x"] * gdf["n_per"]
gdf["y_pop"] = gdf["y"] * gdf["n_per"]

df_pop = gdf.groupby(["COMUNA", "REGION"], as_index=False).agg({
    "x_pop": "sum",
    "y_pop": "sum",
    "n_per": "sum"
})

df_pop["X"] = df_pop["x_pop"] / df_pop["n_per"]
df_pop["Y"] = df_pop["y_pop"] / df_pop["n_per"]

df_pop = df_pop[["COMUNA", "REGION", "X", "Y"]]

# 2. CENTROIDES POBLACIONALES (ENTIDADES)
gdf2["centroid"] = gdf2.geometry.centroid
gdf2["x"] = gdf2["centroid"].x
gdf2["y"] = gdf2["centroid"].y

gdf2["x_pop"] = gdf2["x"] * gdf2["n_per"]
gdf2["y_pop"] = gdf2["y"] * gdf2["n_per"]

df_geom = gdf2.groupby(["COMUNA", "REGION"], as_index=False).agg({
    "x_pop": "sum",
    "y_pop": "sum",
    "n_per": "sum"
})

df_geom["X"] = df_geom["x_pop"] / df_geom["n_per"]
df_geom["Y"] = df_geom["y_pop"] / df_geom["n_per"]

df_geom = df_geom[["COMUNA", "REGION", "X", "Y"]]

# 3. IDENTIFICAR COMUNAS FALTANTES
df_pop["key"]  = list(zip(df_pop["COMUNA"], df_pop["REGION"]))
df_geom["key"] = list(zip(df_geom["COMUNA"], df_geom["REGION"]))

df_geom_faltantes = df_geom[~df_geom["key"].isin(df_pop["key"])].drop(columns="key")
df_pop = df_pop.drop(columns="key")

# 4.  APPEND FINAL
df_final = pd.concat([df_pop, df_geom_faltantes], ignore_index=True)

# 5. GEODATAFRAME FINAL
gdf_centro = gpd.GeoDataFrame(
    df_final,
    geometry=gpd.points_from_xy(df_final["X"], df_final["Y"]),
    crs=gdf.crs
)

print("Comunas desde manzanas:", len(df_pop))
print("Comunas agregadas desde entidades:", len(df_geom_faltantes))
print("Total comunas finales:", gdf_centro[["COMUNA", "REGION"]].drop_duplicates().shape[0])
print("Duplicados:", gdf_centro.duplicated(subset=["COMUNA", "REGION"]).sum())

Comunas desde manzanas: 334
Comunas agregadas desde entidades: 11
Total comunas finales: 345
Duplicados: 0


In [ ]:

gdf_centro_export = gdf_centro.copy()

# asegurar que no sea índice
gdf_centro_export = gdf_centro_export.reset_index(drop=True)

# renombrar columnas
gdf_centro_export = gdf_centro_export.rename(columns={
    "COMUNA": "comuna",
    "X": "x_c",
    "Y": "y_c"
})

# agregar población por comuna (desde manzanas)
df_pob = gdf.groupby("COMUNA", as_index=False)["n_per"].sum().rename(columns={
    "COMUNA": "comuna",
    "n_per": "pob"
})

gdf_centro_export = gdf_centro_export.merge(df_pob, on="comuna", how="left")

# asegurar que comuna sea string (importante para shapefile)
gdf_centro_export["comuna"] = gdf_centro_export["comuna"].astype(str)


gdf_centro_export.to_file(
    "/Users/Angelo/Dropbox/VS codes/final output/nodes_centro_gravedad.shp",
    driver="ESRI Shapefile",
    index=False  # <- clave para no perder la variable comuna
)

## Homologar centroides de gravedad con red actual, 1970 y 1980.

In [86]:
gdf_centro = gdf_centro.to_crs(epsg=32718)
nodes = nodes.to_crs(gdf_centro.crs)
nodes_1970 = nodes_1970.to_crs(gdf_centro.crs)
nodes_1980 = nodes_1980.to_crs(gdf_centro.crs)

Centroide asignado más lejano corresponde al Rio Ibañez, correspondiendo a una distancia de al rededor de 15km hacia el node más cercano, el node que sigue a este corresponde Papudo, cuyo nodo más cercano esta a 6.5 km. La mayoría de los centros de gravedad encuentran su homologo a la red en un rango de 1km (127 no encuentran homologo).

In [87]:
common_nodes = set(nodes_1970["node_ID"]).intersection(
    set(nodes_1980["node_ID"])
)

nodes_1970_common = nodes_1970[
    nodes_1970["node_ID"].isin(common_nodes)
].copy()


join_centroides_1970_common = gpd.sjoin_nearest(
    gdf_centro,
    nodes_1970_common,
    how="left",
    max_distance=10000,
    distance_col="dist"
)

no_match_common = join_centroides_1970_common[
    join_centroides_1970_common["dist"].isna()
]


print("====================================")
print(f"Centroides totales: {len(gdf_centro)}")
print(f"Sin match (nodos comunes): {len(no_match_common)}")
print("====================================")

print(no_match_common["COMUNA_left"].unique())

Centroides totales: 345
Sin match (nodos comunes): 12
['CISNES' 'COCHAMÓ' 'COCHRANE' 'HUALAIHUÉ' 'ISLA DE PASCUA'
 'JUAN FERNÁNDEZ' "O'HIGGINS" 'RÍO IBÁÑEZ' 'LAGO VERDE' 'OLLAGÜE'
 'RÍO VERDE' 'TIMAUKEL']


In [88]:
## Node ID de nodes seleccionados como centroides

join_centroides_1970_common = join_centroides_1970_common.dropna(subset=["index_right"]).copy()

join_centroides_1970_common["node_ID_1970"] = (
    nodes_1970_common.loc[
        join_centroides_1970_common["index_right"].astype(int),
        "node_ID"
    ].values
)

In [89]:
## mapeo de comunas 
map_comuna = (
    join_centroides_1970_common
    .drop_duplicates(subset=["node_ID_1970"])
    .set_index("node_ID")["COMUNA_left"]
)
nodes_1970["COMUNA"] = nodes_1970["node_ID"].map(map_comuna)
nodes_1980["COMUNA"] = nodes_1980["node_ID"].map(map_comuna)
nodes["COMUNA"] = nodes["node_ID"].map(map_comuna)


In [90]:
## Dummy de centro de gravedad poblacional en nodes_1970 (comunes con 1980)
nodes_1970["centro_gravedad_poblacional"] = 0

nodes_1970.loc[
    nodes_1970["node_ID"].isin(join_centroides_1970_common["node_ID_1970"]),
    "centro_gravedad_poblacional"
] = 1

centroid_nodes_1970 = join_centroides_1970_common["node_ID_1970"].unique()
## Lista de centroides
centroid_nodes_1970 = join_centroides_1970_common["node_ID_1970"].unique()


In [91]:
## comprobamos el match
print(len(nodes_1970[nodes_1970["centro_gravedad_poblacional"] == 1]))

329


In [92]:
## Pegar dummy centro de gravedad a node_1980 y node actual
nodes_1980["centro_gravedad_poblacional"] = 0

nodes_1980.loc[
    nodes_1980["node_ID"].isin(centroid_nodes_1970),
    "centro_gravedad_poblacional"
] = 1

## comprobamos el match
print(len(nodes_1980[nodes_1980["centro_gravedad_poblacional"] == 1]))

329


In [93]:
## Pegar dummy centro de gravedad a node actual

nodes["centro_gravedad_poblacional"] = 0

nodes.loc[
    nodes["node_ID"].isin(centroid_nodes_1970),
    "centro_gravedad_poblacional"
] = 1

## comprobamos el match
print(len(nodes[nodes["centro_gravedad_poblacional"] == 1]))

329


In [94]:
# =========================
# 9. GENERAR NODOS CENTRO DE GRAVEDAD EN LA RED Y EDGES DE CONEXIÓN
# =========================
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
import warnings
warnings.filterwarnings('ignore')

# 1. Nuevos nodos
df_centros = join_centroides_1970_common.copy().reset_index(drop=True)

# Aislamos los ID sin colisión
start_id = 1000000
df_centros['nuevo_id'] = range(start_id, start_id + len(df_centros))
df_centros['nodeID'] = df_centros['nuevo_id'].astype(str)
df_centros['node_ID'] = df_centros['nodeID']

nuevos_nodos_list = []
for idx, row in df_centros.iterrows():
    nodo = {
        'node_ID': row['node_ID'],
        'nodeID': row['nodeID'],
        'centro_gravedad_poblacional': 1,
        'COMUNA': row['COMUNA_left'],
        'geometry': row['geometry'],
        'region': row['REGION']
    }
    nuevos_nodos_list.append(nodo)

nuevos_nodos_gdf = gpd.GeoDataFrame(nuevos_nodos_list, geometry='geometry', crs=nodes.crs)

nodes = pd.concat([nodes, nuevos_nodos_gdf], ignore_index=True)
nodes_1970 = pd.concat([nodes_1970, nuevos_nodos_gdf], ignore_index=True)
nodes_1980 = pd.concat([nodes_1980, nuevos_nodos_gdf], ignore_index=True)

# 2. Generar edges de conexión
# Traemos la geometría del nodo homologado usando la llave 'node_ID' única
merged_for_edges = pd.merge(
    df_centros, 
    nodes_1970_common[['node_ID', 'geometry']], 
    left_on='node_ID_1970', 
    right_on='node_ID', 
    suffixes=('_cg', '_red')
)

nuevos_edges_list = []
for idx, row in merged_for_edges.iterrows():
    line = LineString([row['geometry_cg'], row['geometry_red']])
    
    edge = {
        'node_start': row['node_ID_cg'],  
        'node_end': row['node_ID_1970'],  
        'start': row['node_ID_cg'],       # <- CLAVE: Rellenamos start
        'end': row['node_ID_1970'],       # <- CLAVE: Rellenamos end
        'duration': 0,
        'direction': 0,
        'geometry': line
    }
    nuevos_edges_list.append(edge)

nuevos_edges_gdf = gpd.GeoDataFrame(nuevos_edges_list, geometry='geometry', crs=edges.crs)

edges = pd.concat([edges, nuevos_edges_gdf], ignore_index=True)
edges_1970 = pd.concat([edges_1970, nuevos_edges_gdf], ignore_index=True)
edges_1980 = pd.concat([edges_1980, nuevos_edges_gdf], ignore_index=True)

print(f"-> Creados {len(nuevos_nodos_gdf)} nuevos nodos y {len(nuevos_edges_gdf)} nuevos edges.")


-> Creados 334 nuevos nodos y 334 nuevos edges.


In [34]:
# =============================================================================
# 9. CONEXIÓN POR PROYECCIÓN (SPLIT EDGE) DE LOS CENTROS DE GRAVEDAD
# =============================================================================
# import pandas as pd
# import geopandas as gpd
# from shapely.geometry import LineString
# import warnings
# warnings.filterwarnings('ignore')

# def split_edges_for_centroids(centros_df, nodes_df, edges_df, year_prefix=0):
#     print(f"Iniciando Split Edge (Prefijo {year_prefix})...")
    
#     # Encontrar la calle más cercana para cada Centro de Gravedad
#     centros_con_edge = gpd.sjoin_nearest(centros_df, edges_df, how='left', distance_col='dist_to_edge')
    
#     new_nodes = []
#     new_edges = []
#     edges_to_remove = set()
    
#     # ID Inicial para evitar choques con 1970/1980 (Comenzamos en 2 millones)
#     Start_ID = 2000000 + (year_prefix * 1000)
    
#     for idx, centro in centros_con_edge.iterrows():
#         cg_geom = centro.geometry
#         edge_idx = centro['index_right']
#         if pd.isna(edge_idx): continue
        
#         edge_original = edges_df.loc[edge_idx].copy()
#         edge_geom = edge_original.geometry
#         edges_to_remove.add(edge_idx)
        
#         # Proyectar para encontrar el punto de corte en la calle
#         proj_dist = edge_geom.project(cg_geom)
#         point_on_edge = edge_geom.interpolate(proj_dist)
        
#         # --- CREACIÓN DE NODOS ---
#         id_interseccion = str(Start_ID)
#         id_cg = str(Start_ID + 1)
#         Start_ID += 2
        
#         # Rescatamos el nombre de la comuna
#         nom_comuna = centro.get('COMUNA_left', centro.get('COMUNA', 'Desconocida'))
        
#         # Nodo de intersección en la carretera
#         new_nodes.append({
#             'node_ID': id_interseccion, 'nodeID': id_interseccion,
#             'COMUNA': nom_comuna, 'geometry': point_on_edge, 
#             'region': centro.get('REGION_left', centro.get('REGION', 0)),
#             'centro_gravedad_poblacional': 0, 'frontier': 0, 'out_of_net': 0
#         })
        
#         # Nodo del Centro de Gravedad
#         new_nodes.append({
#             'node_ID': id_cg, 'nodeID': id_cg,
#             'COMUNA': nom_comuna, 'geometry': cg_geom, 
#             'region': centro.get('REGION_left', centro.get('REGION', 0)),
#             'centro_gravedad_poblacional': 1, 'frontier': 0, 'out_of_net': 0
#         })
        
#         # --- DIVISIÓN DEL EDGE ---
#         coords = list(edge_geom.coords)
#         seg1_geom = LineString([coords[0], point_on_edge.coords[0]])
#         seg2_geom = LineString([point_on_edge.coords[0], coords[-1]])
        
#         ratio1 = seg1_geom.length / edge_geom.length if edge_geom.length > 0 else 0.5
#         ratio2 = 1 - ratio1
        
#         # Detectar columna de duración
#         d_col = 'duration_n' if 'duration_n' in edge_original else 'duration'
#         dur_orig = float(edge_original[d_col]) if pd.notna(edge_original[d_col]) else 0
        
#         # Parte 1 del edge
#         edge1 = edge_original.to_dict()
#         edge1.update({
#             'edge_ID': f"{edge_original['edge_ID']}_a",
#             'node_end': id_interseccion, 'node_end_f': id_interseccion, 'end': id_interseccion,
#             'geometry': seg1_geom,
#             'length': edge_original['length'] * ratio1,
#             'duration': dur_orig * ratio1,
#             'duration_n': dur_orig * ratio1
#         })
        
#         # Parte 2 del edge
#         edge2 = edge_original.to_dict()
#         edge2.update({
#             'edge_ID': f"{edge_original['edge_ID']}_b",
#             'node_start': id_interseccion, 'node_sta_1': id_interseccion, 'start': id_interseccion,
#             'geometry': seg2_geom,
#             'length': edge_original['length'] * ratio2,
#             'duration': dur_orig * ratio2,
#             'duration_n': dur_orig * ratio2
#         })
        
#         # --- PUENTE CG → RED ---
#         bridge = edge_original.to_dict()
#         bridge.update({
#             'edge_ID': f"bridge_cg_{id_cg}",
#             'node_start': id_cg, 'node_sta_1': id_cg, 'start': id_cg,
#             'node_end': id_interseccion, 'node_end_f': id_interseccion, 'end': id_interseccion,
#             'geometry': LineString([cg_geom, point_on_edge]),
#             'length': cg_geom.distance(point_on_edge),
#             'duration': 0, 'duration_n': 0, 'direction': 0
#         })
        
#         new_edges.extend([edge1, edge2, bridge])

#     # Aplicar cambios
#     edges_clean = edges_df.drop(index=list(edges_to_remove))
    
#     new_edges_gdf = gpd.GeoDataFrame(new_edges, crs=edges_df.crs)
#     new_nodes_gdf = gpd.GeoDataFrame(new_nodes, crs=nodes_df.crs)
    
#     final_edges = pd.concat([edges_clean, new_edges_gdf], ignore_index=True)
#     final_nodes = pd.concat([nodes_df, new_nodes_gdf], ignore_index=True)
    
#     final_nodes['COMUNA'] = final_nodes['COMUNA'].fillna('DESCONOCIDA')
    
#     print(f"-> Operación terminada. Se añadieron {len(new_nodes_gdf)} nodos y {len(new_edges_gdf)} edges.")
#     return final_nodes, final_edges


# =============================================================================
# EJECUCIÓN DEL SPLIT EDGE
# =============================================================================

# df_centros = join_centroides_1970_common.copy().drop(columns=['index_right', 'dist'], errors='ignore').reset_index(drop=True)

# print("Procesando Red Actual...")
# nodes, edges = split_edges_for_centroids(df_centros, nodes, edges, year_prefix=1)

# print("Procesando Red 1970...")
# nodes_1970, edges_1970 = split_edges_for_centroids(df_centros, nodes_1970, edges_1970, year_prefix=2)

# print("Procesando Red 1980...")
# nodes_1980, edges_1980 = split_edges_for_centroids(df_centros, nodes_1980, edges_1980, year_prefix=3)

In [95]:
## Guardar los nodos (Versión 2 - con centros de gravedad integrados)
nodes_1970.to_file('/Users/Angelo/Dropbox/VS codes/final output/nodes_1970_unificado_v2.shp')
nodes_1980.to_file('/Users/Angelo/Dropbox/VS codes/final output/nodes_1980_unificado_v2.shp')
nodes.to_file('/Users/Angelo/Dropbox/VS codes/final output/df_nodes_all_v2.shp')

## Guardar los edges (Versión 2 - con connections a centros de gravedad integrados)
edges_1970.to_file('/Users/Angelo/Dropbox/VS codes/final output/edges_1970_unificado_v2.shp')
edges_1980.to_file('/Users/Angelo/Dropbox/VS codes/final output/edges_1980_1986_unificado_v2.shp')
edges.to_file('/Users/Angelo/Dropbox/VS codes/final output/df_edges_all_v2.shp')

print("¡Archivos exportados exitosamente con sufijo _v2!")


¡Archivos exportados exitosamente con sufijo _v2!
